# Texas Public Improvement District (PID) Model

**Instructions:** Set `INPUTS_PATH` in the cell below to your inputs workbook, then run all cells (`Kernel → Restart & Run All`).

All assumptions are read from the inputs workbook — edit that file to change any input:
- **Inputs** — equivalent tax rate, fees, coupon, CAPI, maturity, reserve method, VTL minimum, sizing switches
- **Developer Information** — lot types (ASP, lot value, absorption) and improvement areas / bond series
- **Overlapping Taxes** — taxing entities and the total-rate cap

The Excel report is written by the last cell.

In [ ]:
# ============================================================
#  SETUP  —  only edit INPUTS_PATH below
# ============================================================
INPUTS_PATH = "inputs/Texas PID Inputs.xlsx"
OUT_DIR     = "output"

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

import pandas as pd
from IPython.display import HTML, display
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_rows = 80

from pid_model.inputs import load_inputs
from pid_model.development import DevelopmentProjection
from pid_model.bonds import BondModel
from pid_model.report import build_excel_report

cfg = load_inputs(INPUTS_PATH)
dev = DevelopmentProjection(cfg).build()
bm  = BondModel(cfg, dev).run()

print(f"{cfg.pid_name}  |  {cfg.city}, {cfg.county} County")
print(f"Equivalent tax rate ${cfg.effective_pid_rate:.4f}  |  coupon {cfg.bond_rate:.3%} + "
      f"{cfg.additional_interest:.2%} additional interest  |  {len(bm.series)} series, {cfg.total_lots} lots")

In [ ]:
# Shared table styling (Tierra palette)
C_HDR_BG, C_HDR_FG = "#1F4E79", "#FFFFFF"
C_TOT_BG, C_TOT_FG = "#BDD7EE", "#1F4E79"
C_SEC_BG, C_SEC_FG = "#D6E4F0", "#1F4E79"
C_ALT_BG, C_WHITE  = "#F2F2F2", "#FFFFFF"

def styled(df, money=(), pct=(), rate=(), x=(), num=()):
    fmt = {}
    fmt.update({c: "${:,.0f}" for c in money})
    fmt.update({c: "{:.2%}" for c in pct})
    fmt.update({c: "${:.4f}" for c in rate})
    fmt.update({c: "{:.2f}x" for c in x})
    fmt.update({c: "{:,.0f}" for c in num})
    return (df.style.hide(axis="index").format(fmt, na_rep="")
            .set_table_styles([
                {"selector": "th", "props": f"background:{C_HDR_BG};color:{C_HDR_FG};"
                                            "font-family:Calibri;font-size:12px;padding:6px 10px;text-align:center"},
                {"selector": "td", "props": "font-family:Calibri;font-size:12px;padding:4px 10px;text-align:right"},
                {"selector": "tr:nth-child(even) td", "props": f"background:{C_ALT_BG}"},
            ]))

def banner(ok, text):
    bg, bd, fg = ("#E2EFDA", "#375623", "#375623") if ok else ("#FFD0D0", "#9C0006", "#9C0006")
    mark = "&#10003;" if ok else "&#9888;"
    display(HTML(f'<div style="background:{bg};border:2px solid {bd};border-radius:6px;padding:10px 16px;'
                 f'font-family:Calibri,sans-serif;font-size:13px;color:{fg};font-weight:bold;margin:6px 0">'
                 f'{mark}&nbsp; {text}</div>'))

---
## Error Check

Green = passes; red = needs attention before the numbers are relied on.

In [ ]:
total_rate = cfg.overlapping_rate + cfg.effective_pid_rate
banner(total_rate <= cfg.max_total_tax_rate + 1e-6,
       f"Total tax rate incl. PID ${total_rate:.4f} vs. cap ${cfg.max_total_tax_rate:.4f} "
       f"(overlapping ${cfg.overlapping_rate:.4f} + PID ${cfg.effective_pid_rate:.4f})")
for s in bm.series:
    banner(s.vtl >= cfg.min_vtl,
           f"{s.series_label} — {s.phase.name}: value-to-lien {s.vtl:.2f}x vs. minimum {cfg.min_vtl:.2f}x")
    short = s.phase.improvement_costs - s.su.project_fund
    banner(short <= 1,
           f"{s.series_label}: project fund ${s.su.project_fund:,.0f} vs. costs ${s.phase.improvement_costs:,.0f}"
           + (f" — shortfall ${short:,.0f} (binding: {s.binding})" if short > 1 else f" (binding: {s.binding})"))
    banner(abs(s.su.total_sources - s.su.total_uses) < 0.01, f"{s.series_label}: sources equal uses")

---
## Summary

In [ ]:
rows = []
for s in bm.series:
    rows.append({"Series": s.series_label, "Improvement Area": s.phase.name, "Delivery": s.delivery,
                 "Lots": s.lots, "Par Amount": s.par, "Binding Constraint": s.binding,
                 "Project Fund": s.su.project_fund, "Improvement Costs": s.phase.improvement_costs,
                 "Value-to-Lien": s.vtl})
summ = pd.DataFrame(rows)
tot = {"Series": "Total", "Lots": summ["Lots"].sum(), "Par Amount": summ["Par Amount"].sum(),
       "Project Fund": summ["Project Fund"].sum(), "Improvement Costs": summ["Improvement Costs"].sum()}
summ = pd.concat([summ, pd.DataFrame([tot])], ignore_index=True)
styled(summ, money=["Par Amount", "Project Fund", "Improvement Costs"], x=["Value-to-Lien"], num=["Lots"])

---
## Sources & Uses

In [ ]:
items = [("Par Amount", "par"), ("Deposit to Project Fund", "project_fund"), ("Reserve Fund", "reserve_fund"),
         ("Delinquency & Prepayment Reserve", "dp_reserve"), ("Capitalized Interest", "capi"),
         ("Administrative Fund", "admin_fund"), ("City Development / PID Fee", "city_fee"),
         ("Underwriter's Discount", "uwd"), ("Costs of Issuance", "coi")]
su = pd.DataFrame({s.series_label: [getattr(s.su, k) for _, k in items] for s in bm.series},
                  index=[l for l, _ in items])
su["Total"] = su.sum(axis=1)
su["% of Par"] = su["Total"] / su.loc["Par Amount", "Total"]
su.loc["Total Uses"] = su.iloc[1:].sum()
su.loc["Total Uses", "% of Par"] = su.loc["Total Uses", "Total"] / su.loc["Par Amount", "Total"]
su = su.reset_index().rename(columns={"index": "Item"})
styled(su, money=[c for c in su.columns if c not in ("Item", "% of Par")], pct=["% of Par"])

---
## Assessment Roll

Every lot type carries the same equivalent tax rate; assessments total to par by series.

In [ ]:
rows = []
for s in bm.series:
    for prod, la in s.lot_assessments.items():
        n = s.phase.lots[prod]
        rows.append({"Series": s.series_label, "Lot Type": prod, "Lots": n,
                     "ASP": la.product.avg_selling_price, "Assessment / Lot": la.assessment,
                     "Total Assessment": la.assessment * n,
                     "First Installment / Lot": la.schedule[0].installment,
                     "Equivalent Rate": la.equivalent_rate})
styled(pd.DataFrame(rows), money=["ASP", "Assessment / Lot", "Total Assessment", "First Installment / Lot"],
       rate=["Equivalent Rate"], num=["Lots"])

---
## Bond Sizing Analysis — Installments vs. Debt Service

In [ ]:
bs = bm.annual_table()
bs = bs[(bs["Total Net Debt Service"] != 0) | (bs["Total Annual Installments"] != 0)]
money = [c for c in bs.columns if c not in ("Bond Year (9/1)", "Assessed Lots", "P&I Coverage")]
styled(bs, money=money, num=["Assessed Lots"], x=["P&I Coverage"])

---
## Per-Lot Annual Installment Schedule

Change `SERIES` / `LOT_TYPE` to view another schedule.

In [ ]:
SERIES, LOT_TYPE = 0, None          # series index; None = first lot type in that series
s = bm.series[SERIES]
la = s.lot_assessments[LOT_TYPE or next(iter(s.lot_assessments))]
print(f"{s.series_label} — {la.product.name}: assessment ${la.assessment:,.0f}, ASP ${la.product.avg_selling_price:,.0f}")
df = la.to_dataframe()
styled(df, money=[c for c in df.columns if c not in ("Year No.", "Payment Date")])

---
## Project and PID Revenue Analysis — Absorption and Taxable Value

In [ ]:
pv = dev.annual_table()
last = pv.loc[pv["Homes Per Year"] > 0, "Year"].max()
pv = pv[pv["Year"] <= last + 2]
styled(pv, money=["Home Value", "Vacant Lot Value", "Total Taxable Basis"],
       num=[c for c in pv.columns if c.endswith("Homes") or c.endswith("Cumulative") or c in
            ("Homes Per Year", "Cumulative Homes")])

---
## Debt Service Schedules

In [ ]:
for s in bm.series:
    display(HTML(f'<div style="background:{C_SEC_BG};color:{C_SEC_FG};font-family:Calibri;font-weight:bold;'
                 f'padding:6px 10px;margin-top:10px">{s.series_label} — {s.phase.name}  |  Par ${s.par:,.0f}</div>'))
    ds = pd.DataFrame([{"Date": r.date, "Principal": r.principal, "Interest": r.interest,
                        "Capitalized Interest": -r.capi, "Net Debt Service": r.net,
                        "Outstanding": r.balance} for r in s.ds])
    display(styled(ds, money=["Principal", "Interest", "Capitalized Interest", "Net Debt Service", "Outstanding"]))

---
## Export to Excel / CSV

In [ ]:
import re
os.makedirs(OUT_DIR, exist_ok=True)
safe = re.sub(r"[^\w\s\-&().]", "", cfg.pid_name).strip()
excel_path = build_excel_report(cfg, dev, bm, os.path.join(OUT_DIR, f"{safe} - PID Analysis.xlsx"))
bm.annual_table().to_csv(f"{OUT_DIR}/bond_sizing_analysis.csv", index=False)
dev.annual_table().to_csv(f"{OUT_DIR}/project_and_pid_rev_analysis.csv", index=False)
print("Excel report:", excel_path)